In [1]:
!pip install prefetch_generator
!pip install tensorboardX

  Preparing metadata (setup.py) ... done
  Created wheel for prefetch_generator: filename=prefetch_generator-1.0.3-py3-none-any.whl size=4758 sha256=81ae7246ac6385832e3bcf3dd495ca53fb7b606551f14a0c2f3addc7e77f7f8f
  Stored in directory: /root/.cache/pip/wheels/9b/08/01/8bacc997ecb83922063bc7dadb42f3cb52cfe43b5217caf820
Successfully built prefetch_generator
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 2.8 MB/s eta 0:00:00


In [2]:
import os
import time
import matplotlib
import numpy as np
from PIL import Image
import torch
from matplotlib import pyplot as plt
from prefetch_generator import BackgroundGenerator
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchvision import transforms, models
from datetime import datetime
from tqdm import tqdm
import torchvision
from torch.optim import Optimizer
from torch.optim.lr_scheduler import LambdaLR
import albumentations as A
from albumentations.pytorch import ToTensorV2
import math
import cv2

# matplotlib.use('Qt5Agg')
matplotlib.rcParams['font.sans-serif'] = ['SimHei']  # 设置支持中文的字体
matplotlib.rcParams['axes.unicode_minus'] = False  # 解决负号显示问题


def seeds(seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


# 示例图
def image_sample(dataloader):
    sample = next(iter(dataloader))
    print(sample["image"].shape, sample["mask"].shape)
    image = sample["image"][0]  # [3,H,W]
    mask = sample["mask"][0]  # [1,H,W]

    image_np = image.permute(1, 2, 0).cpu().numpy()
    mask_np = mask.squeeze(0).cpu().numpy()

    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    image_np = (image_np * std) + mean

    combined = image_np.copy()
    combined[mask_np > 0] = [1, 0, 0]

    titles = ["Original", "Mask", "Original + Mask"]
    image = [image_np, mask_np, combined]

    fig, axs = plt.subplots(1, 3, figsize=(12, 4))
    for ax, img, title in zip(axs, image, titles):
        ax.imshow(img, cmap="gray" if title == "Mask" else None)
        ax.set_title(title)
        ax.axis("off")

    plt.tight_layout()
    plt.show()


class MuralDataset(Dataset):
    def __init__(self, image_dir, mask_dir, size=256):
        self.image_list = sorted([os.path.join(image_dir, f)
                                  for f in os.listdir(image_dir) if f.endswith(('png', 'jpg'))])
        self.mask_list = sorted([os.path.join(mask_dir, f)
                                 for f in os.listdir(mask_dir) if f.endswith(('png', 'jpg'))])
        self.size = size

        assert len(self.image_list) == len(self.mask_list), "图像数量和mask数量不一致！"

        # self.image_transform = transforms.Compose([
        #     transforms.Resize((size, size)),
        #     transforms.ToTensor(),
        # ])

        # self.mask_transform = transforms.Compose([
        #     transforms.Resize((size, size)),
        #     transforms.ToTensor(),
        # ])

        self.transform = A.Compose([
            A.Resize(size, size),

            # 翻转和旋转
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),

            # 模糊处理
            A.GaussianBlur(p=0.5),
            # 亮度对比度调整
            A.RandomBrightnessContrast(p=0.5),
            # 颜色抖动
            A.ColorJitter(p=0.5),

            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ])

    def __len__(self):
        return len(self.image_list)

    def __getitem__(self, idx):
        # image = Image.open(self.image_list[idx]).convert("RGB")
        # mask = Image.open(self.mask_list[idx]).convert("L")

        # image = self.image_transform(image)
        # mask = self.mask_transform(mask)

        image = cv2.imread(self.image_list[idx])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(self.mask_list[idx], 0)

        transformed = self.transform(image=image, mask=mask)
        image = transformed['image']
        mask = transformed['mask']
        mask = (mask > 0).float()  # 二值化掩码
        mask = mask.unsqueeze(0)  # 添加通道维度，变为[1,H,W]

        return {"image": image, "mask": mask}


class DataLoaderX(DataLoader):
    def __iter__(self):
        return BackgroundGenerator(super().__iter__())


class NoiseScheduler(nn.Module):
    def __init__(self, beta_start=0.0001, beta_end=0.02, num_steps=1000):
        """
        Args:
            beta_start: β1，初始噪声水平
            beta_end: βT，最终噪声水平
            num_steps: T，扩散步数
        """
        super().__init__()
        self.beta_start = beta_start
        self.beta_end = beta_end
        self.num_steps = num_steps

        # β_t: 线性噪声调度
        # self.betas 存的是噪声调度表（每一步要加多少噪声）
        self.register_buffer('betas', torch.linspace(beta_start, beta_end, num_steps))
        # α_t = 1 - β_t
        # self.alphas 存的是每一步剩下的“保留率”。
        self.register_buffer('alphas', 1.0 - self.betas)

        # α_bar_t = ∏(1-β_i) from i=1 to t
        # 当前步的累计保留率，即从第1步到第t步，数据还剩下多少比例没有被噪声破坏，累计连乘积会越来越小，原图保留的比例也越来越小
        self.register_buffer('alpha_bar', torch.cumprod(self.alphas, dim=0))

        # 以下两行代码为什么要开方？
        # 因为在扩散模型的数学推导中，噪声的添加和去除涉及到标准差的计算，而标准差是方差的平方根。
        # sqrt(α_bar_t)
        # 控制「保留多少原图」
        self.register_buffer('sqrt_alpha_bar', torch.sqrt(self.alpha_bar))
        # sqrt(1-α_bar_t)
        # 控制「加多少噪声」
        self.register_buffer('sqrt_one_minus_alpha_bar', torch.sqrt(1.0 - self.alpha_bar))

        # ===================== 上面是前向扩散需要的参数 =====================
        # ===================== 下面是反向扩散需要的参数 =====================
        # 1/sqrt(α_bar_t)
        self.register_buffer('sqrt_recip_alphas_bar', torch.sqrt(1.0 / self.alpha_bar))
        # sqrt(1/α_bar_t - 1)
        self.register_buffer('sqrt_recipm1_alphas_bar', torch.sqrt(1.0 / self.alpha_bar - 1))

        # α_bar_(t-1)
        # 前一步的累计保留率
        self.register_buffer('alpha_bar_prev', torch.cat([torch.tensor([1.0]), self.alpha_bar[:-1]], dim=0))
        # 1/sqrt(α_t)
        self.register_buffer('sqrt_recip_alphas', torch.sqrt(1.0 / self.alphas))

        # 后验分布方差 σ_t^2
        self.register_buffer('posterior_var', self.betas * (1.0 - self.alpha_bar_prev) / (1.0 - self.alpha_bar))
        # 后验分布均值系数1: β_t * sqrt(α_bar_(t-1))/(1-α_bar_t)
        self.register_buffer('posterior_mean_coef1',
                             self.betas * torch.sqrt(self.alpha_bar_prev) / (1.0 - self.alpha_bar))
        # 后验分布均值系数2: (1-α_bar_(t-1)) * sqrt(α_t)/(1-α_bar_t)
        self.register_buffer('posterior_mean_coef2',
                             (1.0 - self.alpha_bar_prev) * torch.sqrt(self.alphas) / (1.0 - self.alpha_bar))

    def get(self, var, t, x_shape):
        """获取指定时间步的变量值并调整形状
        Args:
            var: 要查询的变量
            t: 时间步
            x_shape: 目标形状
        Returns:
            调整后的变量值
        """
        # print("t")
        # print(t)

        # 从变量张量中收集指定时间步的值
        out = var[t]

        # print("out")
        # print(out)

        # 调整形状为[batch_size, 1, 1, 1],以便进行广播
        return out.view([t.shape[0]] + [1] * (len(x_shape) - 1))

    def add_noise(self, x, t):
        """向输入添加噪声
        实现公式： x_t = sqrt(α_bar_t) * x_0 + sqrt(1-α_bar_t) * ε, ε ~ N(0,I)
        Args:
            x: 输入图像 x_0
            t: 时间步
        Returns:
            (noisy_x, noise): 加噪后的图像 x_t 和使用噪声 ε
        """
        # 获取时间步 t 对应的 sqrt(α_bar_t)，原图权重
        sqrt_alpha_bar = self.get(self.sqrt_alpha_bar, t, x.shape)
        # 获取时间步 t 对应的 sqrt(1-α_bar_t)，噪声权重
        sqrt_one_minus_alpha_bar = self.get(self.sqrt_one_minus_alpha_bar, t, x.shape)

        # 从标准正态分布中采样噪声 ε~N(0,I)
        noise = torch.randn_like(x)

        # print(sqrt_alpha_bar.shape)
        # print(sqrt_one_minus_alpha_bar.shape)
        # print(x.shape)
        # print(noise.shape)

        # 实现前向扩散过程：x_t = sqrt(α_bar_t) * x_0 + sqrt(1-α_bar_t) * ε
        return sqrt_alpha_bar * x + sqrt_one_minus_alpha_bar * noise, noise


def plot_diffusion_steps(image, noise_scheduler, step_size=100):
    """绘制图像逐步加噪的过程
    Args:
        image: 原始图像
        noise_scheduler: 噪声调度器
        step_size: 每隔多少步绘制一次
    Returns:
        fig: 绘制的图像
    """
    num_images = noise_scheduler.num_steps // step_size
    # 创建一个图形对象，设置画布尺寸为宽15英寸、高3英寸
    fig = plt.figure(figsize=(15, 3))

    # 绘制原始图像
    # 在1行(num_images+1)列的网格中绘制第1个子图（原始图像）
    plt.subplot(1, num_images + 1, 1)
    plt.imshow(image)
    plt.axis('off')
    plt.title('Original')

    # 绘制不同时间步的噪声图像
    for idx in range(num_images):
        t = torch.tensor([idx * step_size])
        # 分别绘制在t时刻的加噪图像
        noisy_image, _ = noise_scheduler.add_noise(image, t)
        plt.subplot(1, num_images + 1, idx + 2)
        plt.imshow(noisy_image)
        plt.axis('off')
        plt.title(f't={t.item()}')

    # 自动调整子图间距
    plt.tight_layout()
    plt.show()
    # return fig


def get_cosine_schedule_with_warmup(
        optimizer: Optimizer,
        num_warmup_steps: int,
        num_training_steps: int,
        num_cycles: float = 0.5,
        last_epoch: int = -1,
):
    """
    Create a schedule with a learning rate that decreases following the values of the cosine function between the
    initial lr set in the optimizer to 0, after a warmup period during which it increases linearly between 0 and the
    initial lr set in the optimizer.
    创建一个计划，其学习率在优化器中设置的初始 lr 与 0 之间的余弦函数值后降低，
    在预热期期间，学习率在 0 和优化器中设置的初始 lr 之间线性增加。

    Args:
        optimizer (:class:`~torch.optim.Optimizer`):
        为其调度学习率的优化器。
        num_warmup_steps (:obj:`int`):
        预热阶段的步骤数。
        num_training_steps (:obj:`int`):
        训练步骤的总数。
        num_cycles (:obj:`float`, `optional`, defaults to 0.5):
        余弦 schedule 中的波数（默认值是在半余弦之后从最大值减少到 0）。
        last_epoch (:obj:`int`, `optional`, defaults to -1):
        恢复训练时最后一个 epoch 的索引。

    Return:
        :obj:`torch.optim.lr_scheduler.LambdaLR` with the appropriate schedule.
    """

    def lr_lambda(current_step):
        # Warmup预热阶段
        if current_step < num_warmup_steps:
            return float(current_step) / float(max(1, num_warmup_steps))
        # t/T 这个比值实际上是一个比例因子，
        # 当t=0（当前步数为0），学习率为0
        # 当t=num_warmup_steps（设定的预热步数），学习率达到设定的最大

        # decadence衰退阶段
        progress = float(current_step - num_warmup_steps) / float(
            max(1, num_training_steps - num_warmup_steps)
        )
        # progress = (t-T)/(n-T)
        # 当t=T的时候，说明已经执行完预热步骤，progress=0
        # 当t=n的时候，说明训练要完成了，progress=1

        return max(
            0.0, 0.5 * (1.0 + math.cos(math.pi * float(num_cycles) * 2.0 * progress))
        )
        # 当progress=0，返回1（比例因子），对应预热阶段结束，衰退阶段开始，最大学习率
        # 当progress=1，返回0（比例因子），对应衰退阶段结束，学习率为0

    return LambdaLR(optimizer, lr_lambda, last_epoch)

2025-11-30 10:28:56.372310: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764498536.578404      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764498536.641291      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
import torch.nn.functional as F
# from sam2.build_sam import build_sam2
# from sam2.sam2_image_predictor import SAM2ImagePredictor
from segment_anything import sam_model_registry, SamPredictor
# from Utils import *


# class SAMAdapterModel(nn.Module):
#     def __init__(self, model_cfg, checkpoint):
#         super().__init__()
#         # 加载官方的模型
#         self.sam = SAM2ImagePredictor(build_sam2(model_cfg, checkpoint))

class Adapter(nn.Module):
    def __init__(self, dim, reduction=4):
        super().__init__()
        hidden_dim = dim // reduction
        self.adapter = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, dim)
        )

    def forward(self, x):
        # print(x.shape)
        return x + self.adapter(x)


class AdapterBlock(nn.Module):
    def __init__(self, block, dim):
        super().__init__()
        self.block = block
        self.adapter = Adapter(dim)

    def forward(self, x):
        out = self.block(x)
        out = self.adapter(out)
        return out


class SAMAdapterModel(nn.Module):
    def __init__(self, checkpoint):
        super().__init__()
        # 加载官方的模型
        self.sam = sam_model_registry["vit_h"](checkpoint=checkpoint)

        self.image_encoder = self.sam.image_encoder
        self.prompt_encoder = self.sam.prompt_encoder
        self.mask_decoder = self.sam.mask_decoder

        # 冻结原始权重
        for p in self.image_encoder.parameters():
            p.requires_grad = False

        # 在每个Transformer block 末尾插 Adapter
        dim = self.image_encoder.blocks[0].norm1.normalized_shape[0]
        self.image_encoder.blocks = nn.ModuleList([
            AdapterBlock(block, dim) for block in self.image_encoder.blocks
        ])

        # 调整位置编码的形状，使其匹配当前输入图像的尺寸
        self. image_encoder.pos_embed = self.resize_pos_embed(
            self.image_encoder.pos_embed, H=512, W=512
        )

    def resize_pos_embed(self, pos_embed, H, W):
        # 获取原始位置编码的高度和宽度
        # [1, 64, 64, 768]
        # 原始的位置编码是针对 1024x1024 图像设计的，经过 patch embedding 后（分割成16×16的小方块，1024÷16）变为 64x64，
        # 每个 16×16 patch 都有自己的一条 768 维的位置向量
        _, H_old, W_old, C = pos_embed.shape

        # 计算新的高度和宽度，因为输入别的尺寸的图像，匹配不了原来的patch
        # 512//16 512//16
        H_new, W_new = H // 16, W // 16

        pos_embed = pos_embed.permute(0, 3, 1, 2)  # [1, C, H_old, W_old]

        # 把预训练的二维位置编码当作多通道图片，用双线性插值的方法重新采样到目标分辨率
        pos_embed_resized = F.interpolate(pos_embed, size=(H_new, W_new), mode='bilinear', align_corners=False)

        # [1, 32, 32, 768]
        pos_embed_resized = pos_embed_resized.permute(0, 2, 3, 1)  # [1, H_new, W_new, C]

        # 把一个普通张量包装成可训练的模型参数并返回
        return nn.Parameter(pos_embed_resized)

    def forward(self, images):
        # [16, 3, 512, 512]
        B, C, H, W = images.shape

        # # 调整位置编码的形状，使其匹配当前输入图像的尺寸
        # self.image_encoder.pos_embed = self.resize_pos_embed(self.image_encoder.pos_embed, H, W)

        # SAM image_encoder 提取图像特征
        features = self.image_encoder(images)  # [B, C, H/16, W/16]
        # print(features.shape)

        # print(self.prompt_encoder)
        # SAM mask_decoder 需要 prompt embedding；这里直接给空（全图分割）
        sparse_embeddings, dense_embeddings = self.prompt_encoder(
            points=None,
            boxes=None,
            masks=None,
        )
        image_pe = self.prompt_encoder.get_dense_pe()

        # print(image_pe.shape)
        # print(sparse_embeddings.shape)
        # print(dense_embeddings.shape)

        H_feat = features.shape[-2]
        W_feat = features.shape[-1]
        image_pe = F.interpolate(image_pe, size=(H_feat, W_feat), mode='bilinear', align_corners=False)
        dense_embeddings = F.interpolate(dense_embeddings, size=(H_feat, W_feat), mode='bilinear', align_corners=False)

        # print(image_pe.shape)
        # print(dense_embeddings.shape)
        #
        # print("=======================================")

        mask, _ = self.mask_decoder(
            image_embeddings=features,
            image_pe=image_pe,
            sparse_prompt_embeddings=sparse_embeddings,
            dense_prompt_embeddings=dense_embeddings,
            multimask_output=False
        )

        mask = F.interpolate(
            mask, size=(H, W),  # 这里的 H, W 就是原图 512, 512
            mode="bilinear", align_corners=False
        )
        return mask


class BCEDiceLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, preds, targets):
        bce_loss = self.bce(preds, targets)

        preds = torch.sigmoid(preds)
        # 计算每个样本的交集和并集
        # 交集
        # (preds * targets) 表示 预测与真实的重叠区域
        # 然后对 [C,H,W] 这三个维度求和，每个 batch 样本得到一个总的“交集面积”
        intersection = (preds * targets).sum(dim=(1, 2, 3))
        # 并集
        # preds.sum(...)：预测的“前景面积”（概率加总）。
        # targets.sum(...)：真实标签的前景面积（像素值 0/1 加总）。
        # 两者相加 ≈ 预测面积 + 真实面积。
        union = preds.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))

        # 计算每个样本的 Dice 系数并取平均
        # 2 * intersection 是 2×|A ∩ B| + ε
        # union + epsilon 是 |A|+|B| + ε
        epsilon = 1e-5
        dice_loss = 1 - (2. * intersection + epsilon) / (union + epsilon)
        dice = dice_loss.mean()

        return bce_loss + dice

# 若更在意召回（减少漏检），把 beta 调大（例如 alpha=0.3, beta=0.7）。
# 若更在意精度（减少误报），把 alpha 调大。
# gamma 常在 [0.5, 2.0] 范围内试验：越大更专注难样本，但也可能导致训练不稳定。
# 常把 FocalTversky 与其他损失（比如 BCE）结合使用，特别是在早期训练阶段，这有助于稳定梯度。
class BoundaryLoss(nn.Module):
    pass


class FocalTversky_BCE_Loss(nn.Module):
    def __init__(self, alpha=0.3, beta=0.7, gamma=0.5):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma

        self.BCE = nn.BCEWithLogitsLoss()
        # self.Boundary = BoundaryLoss()

    def forward(self, preds, targets):
        bce_loss = self.BCE(preds, targets)

        # boundary_loss = self.Boundary(preds, targets)

        preds = torch.sigmoid(preds)

        smooth = 1e-5
        # preds = preds.view(-1)
        # targets = targets.view(-1)

        # 如果 target 是 1，pred 是 1：
        # 这是 TP（预测正确）。
        TP = (preds * targets).sum(dim=(1, 2, 3))
        # 如果 target 为 0（背景），pred 为 1：
        # 这是 FP（误检）。
        FP = ((1 - targets) * preds).sum(dim=(1, 2, 3))
        # 如果 target 为 1，但 pred 为 0：
        # 这是 FN（漏检）。
        FN = (targets * (1 - preds)).sum(dim=(1, 2, 3))

        Tversky = (TP + smooth) / (TP + self.alpha * FP + self.beta * FN + smooth)
        focal_tversky = (1 - Tversky).pow(self.gamma).mean()
        return 0.3 * bce_loss + 0.7 * focal_tversky

In [4]:
# def train_step(model, train_loader, criterion, optimizer, scaler, epoch, num_epochs, device):
#     model.train()
#     train_losses = 0.0

#     train_loader_tqdm = tqdm(train_loader, desc=f"Train Epoch {epoch + 1}/{num_epochs}", leave=True)
#     for batch in train_loader_tqdm:
#         # 获取一个batch的图像数据并移至指定设备
#         # [16, 3, 256, 256]
#         images = batch["image"].to(device)
#         masks = batch["mask"].to(device)

#         with torch.amp.autocast(device_type='cuda'):
#             # 使用模型预测噪声
#             preds = model(images)

#             # print(preds.shape, masks.shape)
#             # time.sleep(60)

#             # 计算预测噪声与真实噪声之间的损失
#             loss = criterion(preds, masks)

#         optimizer.zero_grad()
#         scaler.scale(loss).backward()
#         scaler.unscale_(optimizer)
#         torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
#         scaler.step(optimizer)
#         scaler.update()

#         # # 反向传播和优化
#         # optimizer.zero_grad()  # 清空梯度
#         # loss.backward()  # 反向传播计算梯度
#         # torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # 梯度裁剪，防止梯度爆炸
#         # optimizer.step()  # 更新参数

#         # 累计损失
#         train_losses += loss.item()
#         # 显示当前轮次的累计损失
#         train_loader_tqdm.set_postfix(loss=train_losses / len(train_loader))

#     return train_losses / len(train_loader)

In [5]:
def train_step(model, train_loader, criterion, optimizer, scaler, epoch, num_epochs, device, accum_steps=2):
    model.train()
    train_losses = 0.0
    optimizer.zero_grad(set_to_none=True)  # 只在外层先清空一次梯度

    train_loader_tqdm = tqdm(train_loader, desc=f"Train Epoch {epoch + 1}/{num_epochs}", leave=True)
    for step, batch in enumerate(train_loader_tqdm):
        # 获取一个batch的图像数据并移至指定设备
        # [16, 3, 256, 256]
        images = batch["image"].to(device)
        masks = batch["mask"].to(device)

        with torch.amp.autocast(device_type='cuda'):
            # 使用模型预测噪声
            preds = model(images)

            # print(preds.shape, masks.shape)
            # time.sleep(60)

            # 计算预测噪声与真实噪声之间的损失
            loss = criterion(preds, masks)

            loss = loss / accum_steps  # 梯度累积，损失也要相应缩小

        scaler.scale(loss).backward()

        # 每 accum_steps 个 小batch 更新一次参数
        if (step + 1) % accum_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        # # 反向传播和优化
        # optimizer.zero_grad()  # 清空梯度
        # loss.backward()  # 反向传播计算梯度
        # torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # 梯度裁剪，防止梯度爆炸
        # optimizer.step()  # 更新参数

        # 累计损失
        train_losses += loss.item() * accum_steps  # 还原回正常的损失值
        # 显示当前轮次的累计损失
        train_loader_tqdm.set_postfix(loss=train_losses / (step + 1))

    # # 如果最后一个小batch没凑够accum_steps，也要更新一次参数
    # if (step+1) % accum_steps != 0:
    #     scaler.unscale_(optimizer)
    #     torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    #     scaler.step(optimizer)
    #     scaler.update()
    #     optimizer.zero_grad(set_to_none=True)

    return train_losses / len(train_loader)


def test_step(model, test_loader, criterion, epoch, num_epochs, device):
    model.eval()
    test_losses = 0.0
    total_iou = 0.0
    total_images = 0

    with torch.no_grad():
        test_loader_tqdm = tqdm(test_loader, desc=f"Test Epoch {epoch + 1}/{num_epochs}", leave=True)
        for step, batch in enumerate(test_loader_tqdm):
            images = batch["image"].to(device)
            masks = batch["mask"].to(device)
            # masks = (masks > 0.5).float()

            preds = model(images)
            loss = criterion(preds, masks)
            test_losses += loss.item()

            preds = torch.sigmoid(preds)
            preds = (preds > 0.5).float()

            # IoU = |A∩B| / |A∪B|
            #     = |A∩B| / (|A|+|B|−|A∩B|)
            # ∣A∩B∣
            intersection = (preds * masks).sum(dim=(1, 2, 3))
            # |A|+|B|
            union = preds.sum(dim=(1, 2, 3)) + masks.sum(dim=(1, 2, 3))
            iou_per_image = (intersection + 1e-6) / (union - intersection + 1e-6)

            total_iou += iou_per_image.sum().item()
            total_images += images.size(0)

            test_loader_tqdm.set_postfix(loss=test_losses / (step + 1),
                                         iou=total_iou / total_images)

    return total_iou / total_images, test_losses / len(test_loader)


def train_model(model, train_loader, test_loader, criterion, optimizer, scheduler, scaler, device, num_epochs=100):
    os.makedirs("./model", exist_ok=True)
    os.makedirs("./model_step", exist_ok=True)
    writer = SummaryWriter(log_dir=f"./logs/Seg_{datetime.now().strftime('%Y%m%d_%H%M%S')}")
    best_train_loss = float('inf')
    best_iou = 0.0
    best_test_loss = float('inf')
    
    for epoch in range(num_epochs):
        train_loss = train_step(model, train_loader, criterion, optimizer, scaler, epoch, num_epochs, device)
        iou, test_loss = test_step(model, test_loader, criterion, epoch, num_epochs, device)

        writer.add_scalar('Loss/Train', train_loss, epoch)
        writer.add_scalar('Loss/Test', test_loss, epoch)
        writer.add_scalar('IOU/Test', iou, epoch)
        writer.add_scalar('Learning Rate', optimizer.param_groups[0]['lr'], epoch)

        scheduler.step()

        if iou > best_iou:
            best_iou = iou
            torch.save(model.state_dict(), f"./model/best_iou_model.pth")
            print(f"\nBest Test Model Saved at Epoch {epoch + 1}")
        if test_loss < best_test_loss:
            best_test_loss = test_loss
            torch.save(model.state_dict(), f"./model/best_test_model.pth")
            print(f"\nBest Test Loss Model Saved at Epoch {epoch + 1}")
            
    # 保存最后一轮的模型
    torch.save(model.state_dict(), f"./model/last_epoch_model.pth")
    print(f"\nLast Epoch Model Saved after {num_epochs} epochs")

In [6]:
def train_and_val(model, train_loader, val_loader, criterion, optimizer, scheduler, scaler, device, 
                  total_steps, valid_steps, accum_steps):
    os.makedirs("./model", exist_ok=True)
    os.makedirs("./model_step", exist_ok=True)
    writer = SummaryWriter(log_dir=f"./logs/Seg_{datetime.now().strftime('%Y%m%d_%H%M%S')}")
    best_train_loss = float('inf')
    best_val_loss = float('inf')
    best_iou = 0.0

    train_iterator = iter(train_loader)
    train_loader_tqdm = tqdm(range(total_steps), desc="Train", leave=False)

    for step in train_loader_tqdm:
        model.train()

        try:
            batch = next(train_iterator)
        except StopIteration:
            train_iterator = iter(train_loader)
            batch = next(train_iterator)

        # 获取一个batch的图像数据并移至指定设备
        # [16, 3, 256, 256]
        images = batch["image"].to(device)
        masks = batch["mask"].to(device)

        with torch.amp.autocast(device_type='cuda'):
            # 使用模型预测噪声
            preds = model(images)

            # print(preds.shape, masks.shape)
            # time.sleep(60)

            # 计算预测噪声与真实噪声之间的损失
            loss = criterion(preds, masks)

            loss = loss / accum_steps  # 梯度累积，损失也要相应缩小

        scaler.scale(loss).backward()

        # 每 accum_steps 个 小batch 更新一次参数
        if (step + 1) % accum_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        # # 反向传播和优化
        # optimizer.zero_grad()  # 清空梯度
        # loss.backward()  # 反向传播计算梯度
        # torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # 梯度裁剪，防止梯度爆炸
        # optimizer.step()  # 更新参数

        # 累计损失
        train_losses = loss.item() * accum_steps  # 还原回正常的损失值
        # 显示当前轮次的累计损失
        train_loader_tqdm.set_postfix(loss=train_losses, step=step + 1)
        writer.add_scalar('Loss/Train', train_losses, step + 1)
        writer.add_scalar('Learning Rate', optimizer.param_groups[0]['lr'], step + 1)

        if (step + 1) % valid_steps == 0:
            model.eval()
            val_losses = 0.0

            total_iou = 0.0
            total_images = 0

            val_loader_tqdm = tqdm(val_loader, desc="Valid", leave=False)
            with torch.no_grad():
                for i, batch in enumerate(val_loader_tqdm):
                    images = batch["image"].to(device)
                    masks = batch["mask"].to(device)

                    preds = model(images)
                    loss = criterion(preds, masks)
                    val_losses += loss.item()

                    preds = torch.sigmoid(preds)
                    preds = (preds > 0.5).float()

                    # IoU = |A∩B| / |A∪B|
                    #     = |A∩B| / (|A|+|B|−|A∩B|)
                    # ∣A∩B∣
                    intersection = (preds * masks).sum(dim=(1, 2, 3))
                    # |A|+|B|
                    union = preds.sum(dim=(1, 2, 3)) + masks.sum(dim=(1, 2, 3))
                    # IoU
                    iou_per_image = (intersection + 1e-6) / (union - intersection + 1e-6)

                    total_iou += iou_per_image.sum().item()
                    total_images += images.size(0)

                    iou = total_iou / total_images

                    val_loader_tqdm.set_postfix(loss=val_losses / (i + 1), iou=iou, batch=i+1)

            model.train()

            val_loss = val_losses / len(val_loader)

            writer.add_scalar('Loss/Val', val_loss, step + 1)
            writer.add_scalar('IOU/Val', iou, step + 1)

            if iou > best_iou:
                torch.save(model.state_dict(), f"./model/best_iou_model.pth")
                best_iou = iou
                print(f"\nBest IoU Model Saved at Epoch {step + 1}")

            if val_loss < best_val_loss:
                torch.save(model.state_dict(), f"./model/best_val_model.pth")
                best_val_loss = val_loss
                print(f"\nBest Val Loss Model Saved at Epoch {step + 1}")
        
        # # 每1000步保存一次模型
        # if (step + 1) % 1000 == 0:
        #     torch.save(model.state_dict(), f"./model_step/model_step_{step + 1}.pth")
        #     print(f"\nModel Saved at Step {step + 1}")
    
    torch.save(model.state_dict(), f"./model/last_model.pth")
    print(f"Last Model Saved")
    
    writer.close()

In [7]:
# from tensorboardX import SummaryWriter
# writer = SummaryWriter(log_dir=f"./logs/Seg_{datetime.now().strftime('%Y%m%d_%H%M%S')}")
# writer.close()

In [8]:
# %reload_ext tensorboard
# %tensorboard --logdir /kaggle/working/logs

In [9]:
# !tensorboard --logdir='./ImageSeg/logs'

In [10]:
if __name__ == "__main__":
    seeds(125)

    # 检查当前系统是否有可用的 GPU
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"Training on {device}")

    train_image_dir = "/kaggle/input/mural-seg/Mural_seg/train/images"
    train_mask_dir = "/kaggle/input/mural-seg/Mural_seg/train/labels"
    test_image_dir = "/kaggle/input/mural-seg/Mural_seg/test/images"
    test_mask_dir = "/kaggle/input/mural-seg/Mural_seg/test/labels"

    train_dataset = MuralDataset(train_image_dir, train_mask_dir, size=512)
    train_dataloader = DataLoaderX(
        train_dataset,
        batch_size=4,
        shuffle=True,
        drop_last=True,
        num_workers=4,
        pin_memory=True,
        persistent_workers=True,
    )
    test_dataset = MuralDataset(test_image_dir, test_mask_dir, size=512)
    test_dataloader = DataLoaderX(
        test_dataset,
        batch_size=4,
        shuffle=False,
        drop_last=False,
        num_workers=4,
        pin_memory=True,
        persistent_workers=True,
    )
    # 查看dataloader数量
    print(len(train_dataloader))
    print(len(test_dataloader))
    num_epochs = 50
    accum_steps = 2
    total_steps = 3000

    scheduler_steps = int(total_steps // accum_steps)
    warmup_steps = int(0.1 * scheduler_steps)

    valid_steps = 25

    
    # sam_model_path = r'./model/sam2.1_hiera_large.pt'
    # sam_model_cfg_path = r'./model/sam2.1_hiera_l.yaml'
    # model = SAMAdapterModel(model_cfg=sam_model_cfg_path, checkpoint=sam_model_path)

    sam_model_path = r'/kaggle/input/segment-anything/pytorch/vit-h/1/model.pth'
    # sam_model_path = r'./model/sam_vit_h_4b8939.pth'

    model = SAMAdapterModel(checkpoint=sam_model_path)
    print(model)
    model = model.to(device)
    
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=2e-4, weight_decay=1e-5)
    # criterion = BCEDiceLoss()
    criterion = FocalTversky_BCE_Loss()
    # scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.7)
    scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, scheduler_steps)
    scaler = torch.amp.GradScaler()  # 混合精度

    # train_model(model, train_dataloader, test_dataloader, criterion, optimizer, scheduler, scaler, device, num_epochs=25)
    train_and_val(model, train_dataloader, test_dataloader, criterion, optimizer, scheduler, scaler, device,
                  total_steps=total_steps, valid_steps=valid_steps, accum_steps=accum_steps)

Training on cuda:0
190
51
SAMAdapterModel(
  (sam): Sam(
    (image_encoder): ImageEncoderViT(
      (patch_embed): PatchEmbed(
        (proj): Conv2d(3, 1280, kernel_size=(16, 16), stride=(16, 16))
      )
      (blocks): ModuleList(
        (0-31): 32 x AdapterBlock(
          (block): Block(
            (norm1): LayerNorm((1280,), eps=1e-06, elementwise_affine=True)
            (attn): Attention(
              (qkv): Linear(in_features=1280, out_features=3840, bias=True)
              (proj): Linear(in_features=1280, out_features=1280, bias=True)
            )
            (norm2): LayerNorm((1280,), eps=1e-06, elementwise_affine=True)
            (mlp): MLPBlock(
              (lin1): Linear(in_features=1280, out_features=5120, bias=True)
              (lin2): Linear(in_features=5120, out_features=1280, bias=True)
              (act): GELU(approximate='none')
            )
          )
          (adapter): Adapter(
            (adapter): Sequential(
              (0): Linear(in_featu

Valid:  76%|███████▋  | 39/51 [00:46<00:14,  1.19s/it, batch=39, iou=0.0138, loss=0.768]libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile

Valid:  78%|███████▊  | 40/51 [00:48<00:13,  1.19s/it, batch=40, iou=0.0136, loss=0.768]libpng warning: iCCP: known incorrect sRGB profile

Valid: 100%|██████████| 51/51 [01:00<00:00,  1.05it/s, batch=51, iou=0.0109, loss=0.769]
                                                                                        


Best IoU Model Saved at Epoch 25


Train:   1%|          | 25/3000 [02:09<18:42:29, 22.64s/it, loss=0.761, step=25]


Best Val Loss Model Saved at Epoch 25


...


Best IoU Model Saved at Epoch 2750


Train:  92%|█████████▏| 2750/3000 [3:45:09<1:38:24, 23.62s/it, loss=0.445, step=2750]


Best Val Loss Model Saved at Epoch 2750


Valid:  76%|███████▋  | 39/51 [00:47<00:14,  1.20s/it, batch=39, iou=0.323, loss=0.573]libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile

Valid:  78%|███████▊  | 40/51 [00:48<00:13,  1.20s/it, batch=40, iou=0.32, loss=0.575]libpng warning: iCCP: known incorrect sRGB profile

Valid:  76%|███████▋  | 39/51 [00:47<00:14,  1.20s/it, batch=39, iou=0.317, loss=0.579]libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile

Valid:  78%|███████▊  | 40/51 [00:48<00:13,  1.20s/it, batch=40, iou=0.314, loss=0.58]libpng warning: iCCP: known incorrect sRGB profile

Train:  93%|█████████▎| 2803/3000 [3:49:19<28:25,  8.66s/it, loss=0.436, step=2803]libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
Valid:  76%|███████▋  | 39/51 [00:47<00:14,  1.20s/it, ba

Last Model Saved


In [11]:
from IPython.display import FileLink
FileLink('model/best_iou_model.pth')
FileLink('model/best_val_model.pth')
FileLink('model/last_model.pth')

/kaggle/working/model/last_model.pth

In [12]:
!zip -r /kaggle/working/working.zip /kaggle/working
!zip -r /kaggle/working/logs.zip /kaggle/working

  adding: kaggle/working/ (stored 0%)
  adding: kaggle/working/model_step/ (stored 0%)
  adding: kaggle/working/model/ (stored 0%)
  adding: kaggle/working/model/last_model.pth (deflated 7%)
  adding: kaggle/working/model/best_val_model.pth (deflated 7%)
  adding: kaggle/working/model/best_iou_model.pth (deflated 7%)
  adding: kaggle/working/__notebook__.ipynb (deflated 96%)
  adding: kaggle/working/logs/ (stored 0%)
  adding: kaggle/working/logs/Seg_20251130_102934/ (stored 0%)
  adding: kaggle/working/logs/Seg_20251130_102934/events.out.tfevents.1764498574.32540735ebd6.19.0 (deflated 69%)
  adding: kaggle/working/ (stored 0%)
  adding: kaggle/working/model_step/ (stored 0%)
  adding: kaggle/working/model/ (stored 0%)
  adding: kaggle/working/model/last_model.pth (deflated 7%)
  adding: kaggle/working/model/best_val_model.pth (deflated 7%)
  adding: kaggle/working/model/best_iou_model.pth
zip I/O error: No space left on device
zip error: Output file write failure (write error on zip f

In [13]:
FileLink('working.zip')
FileLink('logs.zip')

/kaggle/working/logs.zip